In [1]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, brier_score_loss
from sklearn.calibration import CalibratedClassifierCV
from agent_tools import fetch_ulb_fraud_data

In [2]:
print("📥 [Step 1] Streaming production ULB records via cloud pipeline...")
df = fetch_ulb_fraud_data()

📥 [Step 1] Streaming production ULB records via cloud pipeline...
⏳ 正在建立安全连接并从 BigQuery 提取 ULB 数据...
✅ 数据提取成功！共成功加载 284807 条风控样本数据。


In [3]:
# Cast NUMERIC fields to native float64 types to safely interface with native C++ matrixes
if 'amount' in df.columns:
    df['amount'] = df['amount'].astype(float)
    
X = df.drop(columns=['class', 'time']).fillna(0)
y = df['class'].astype(int)

In [4]:
X.shape

(284807, 29)

In [5]:
y.shape

(284807,)

In [8]:
int(np.sum(y))/y.shape[0]

0.001727485630620034

In [9]:
# -------------------------------------------------------------------------
# 🧱 DATA SPLITTING (FinTech Stratification Protocol)
# -------------------------------------------------------------------------
# In fraud, we MUST use stratify=y. This guarantees that train, test, and
# calibration datasets maintain the exact same 0.17% rare-event ratio.
# -------------------------------------------------------------------------
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
# Split training sets down further to isolate an independent Calibration pool
X_train, X_calib, y_train, y_calib = train_test_split(
    X_train_val, y_train_val, test_size=0.25, random_state=42, stratify=y_train_val
)

print(f"✅ Data Stratified: Train={X_train.shape[0]}, Calib={X_calib.shape[0]}, Test={X_test.shape[0]}")

✅ Data Stratified: Train=170883, Calib=56962, Test=56962


In [10]:
print(f"✅ Data Stratified: Train={y_train.shape[0]}, Calib={y_calib.shape[0]}, Test={y_test.shape[0]}")

✅ Data Stratified: Train=170883, Calib=56962, Test=56962


In [11]:
# -------------------------------------------------------------------------
# 🏋️‍♂️ BASE XGBOOST MODEL WITH RISK TUNED CLASS-WEIGHTS
# -------------------------------------------------------------------------
# scale_pos_weight forces the booster to penalize missing rare positive labels
imbalance_ratio = (len(y_train) - sum(y_train)) / sum(y_train)

In [12]:
imbalance_ratio

578.264406779661

In [13]:
print(f"⚖️ Calculated Optimal Imbalance Class Scale Weight Factor: {imbalance_ratio:.2f}")

⚖️ Calculated Optimal Imbalance Class Scale Weight Factor: 578.26


In [14]:
base_model = xgb.XGBClassifier(
    max_depth=5,
    learning_rate=0.1,
    n_estimators=100,
    scale_pos_weight=imbalance_ratio,
    eval_metric='logloss',
    random_state=42
)

print("🌲 Fitting uncalibrated baseline gradient-boosted arrays...")
base_model.fit(X_train, y_train)

🌲 Fitting uncalibrated baseline gradient-boosted arrays...


,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",'logloss'
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [15]:
# -------------------------------------------------------------------------
# 🎯 PROBABILITY CALIBRATION LAYER (Isotonic Wrapper)
# -------------------------------------------------------------------------
# cv='prefit' means we use the already-trained model and calibrate it
# using the isolated X_calib set to completely bypass data-leakage bugs.
# -------------------------------------------------------------------------
print("🎯 Initiating Isotonic Regression probability calibration sequence...")
calibrated_model = CalibratedClassifierCV(
    estimator=base_model,
    method='isotonic',
    ensemble=False
)
calibrated_model.fit(X_calib, y_calib)

🎯 Initiating Isotonic Regression probability calibration sequence...


,"estimator estimator: estimator instance, default=NoneThe classifier whose output need to be calibrated to provide moreaccurate `predict_proba` outputs. The default classifier isa :class:`~sklearn.svm.LinearSVC`... versionadded:: 1.2","XGBClassifier...ree=None, ...)"
,"method method: {'sigmoid', 'isotonic', 'temperature'}, default='sigmoid'The method to use for calibration. Can be:- 'sigmoid', which corresponds to Platt's method (i.e. a binary logistic regression model).- 'isotonic', which is a non-parametric approach.- 'temperature', temperature scaling.Sigmoid and isotonic calibration methods natively support only binaryclassifiers and extend to multi-class classification using a One-vs-Rest (OvR)strategy with post-hoc renormalization, i.e., adjusting the probabilities aftercalibration to ensure they sum up to 1.In contrast, temperature scaling naturally supports multi-class calibration byapplying `softmax(classifier_logits/T)` with a value of `T` (temperature)that optimizes the log loss.For very uncalibrated classifiers on very imbalanced datasets, sigmoidcalibration might be preferred because it fits an additional interceptparameter. This helps shift decision boundaries appropriately when theclassifier being calibrated is biased towards the majority class.Isotonic calibration is not recommended when the number of calibration samplesis too low ``(≪1000)`` since it then tends to overfit... versionchanged:: 1.8 Added option 'temperature'.",'isotonic'
,"ensemble ensemble: bool, or ""auto"", default=""auto""Determines how the calibrator is fitted.""auto"" will use `False` if the `estimator` is a:class:`~sklearn.frozen.FrozenEstimator`, and `True` otherwise.If `True`, the `estimator` is fitted using training data, andcalibrated using testing data, for each `cv` fold. The final estimatoris an ensemble of `n_cv` fitted classifier and calibrator pairs, where`n_cv` is the number of cross-validation folds. The output is theaverage predicted probabilities of all pairs.If `False`, `cv` is used to compute unbiased predictions, via:func:`~sklearn.model_selection.cross_val_predict`, which are thenused for calibration. At prediction time, the classifier used is the`estimator` trained on all the data.Note that this method is also internally implemented in:mod:`sklearn.svm` estimators with the `probabilities=True` parameter... versionadded:: 0.24.. versionchanged:: 1.6 `""auto""` option is added and is the default.",False
,"cv cv: int, cross-validation generator, or iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross-validation,- integer, to specify the number of folds,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if ``y`` is binary or multiclass,:class:`~sklearn.model_selection.StratifiedKFold` is used. If ``y`` isneither binary nor multiclass, :class:`~sklearn.model_selection.KFold`is used.Refer to the :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",None
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors.Base estimator clones are fitted in parallel across cross-validationiterations.See :term:`Glossary <n_jobs>` for more details... versionadded:: 0.24",None
Name,Type,Value
"calibrated_classifiers_ calibrated_classifiers_: list (len() equal to cv or 1 if `ensemble=False`)The list of classifier and calibrator pairs.- When `ensemble=True`, `n_cv` fitted `estimator` and calibrator pairs. `n_cv` is the number of cross-validation folds.- When `ensemble=False`, the `estimator`, fitted on all the data, and fitted calibrator... versionchanged:: 0.24 Single calibrated classifier case when `ensemble=False`.",list,[<sklearn.cali...x7f42a4ddcc80>]

In [16]:
# -------------------------------------------------------------------------
# 📊 FINTECH RISK EVALUATION METRICS
# -------------------------------------------------------------------------
# Brier Score Loss calculates the direct mean squared error of probabilities.
# Lower Brier score = highly accurate probability tracking!
# -------------------------------------------------------------------------
raw_probs = base_model.predict_proba(X_test)[:, 1]
calib_probs = calibrated_model.predict_proba(X_test)[:, 1]

In [17]:
print("\n================== 📊 MODEL EVALUATION CARD ==================")
print(f"📉 Raw Uncalibrated Brier Score Error: {brier_score_loss(y_test, raw_probs):.5f}")
print(f"🔥 Calibrated Stable Brier Score Error: {brier_score_loss(y_test, calib_probs):.5f}")


================== 📊 MODEL EVALUATION CARD ==================
📉 Raw Uncalibrated Brier Score Error: 0.00060
🔥 Calibrated Stable Brier Score Error: 0.00053


In [20]:
# ------------------------------------------------------------------------- 
# 📊 FINTECH RISK EVALUATION METRICS 
# ------------------------------------------------------------------------- 
raw_probs = base_model.predict_proba(X_test)[:, 1] 
calib_probs = calibrated_model.predict_proba(X_test)[:, 1] 

print("\n================== 📊 MODEL EVALUATION CARD ==================") 
print(f"📉 Raw Uncalibrated Brier Score Error: {brier_score_loss(y_test, raw_probs):.5f}") 
print(f"🔥 Calibrated Stable Brier Score Error: {brier_score_loss(y_test, calib_probs):.5f}") 


================== 📊 MODEL EVALUATION CARD ==================
📉 Raw Uncalibrated Brier Score Error: 0.00060
🔥 Calibrated Stable Brier Score Error: 0.00053


In [21]:
# 🛠️ ADJUST THRESHOLD TO 0.20 TO CATCH MORE FRAUD
custom_threshold = 0.20
predictions = (calib_probs >= custom_threshold).astype(int)

In [22]:
type(predictions)

numpy.ndarray

In [23]:
predictions.shape

(56962,)

In [24]:
print(f"\n📋 Production Classification Profile (Threshold = {custom_threshold}):") 
print(classification_report(y_test, predictions, target_names=['Legit', 'Fraud'])) 
print("==============================================================") 


📋 Production Classification Profile (Threshold = 0.2):
              precision    recall  f1-score   support

       Legit       1.00      1.00      1.00     56864
       Fraud       0.81      0.81      0.81        98

    accuracy                           1.00     56962
   macro avg       0.90      0.90      0.90     56962
weighted avg       1.00      1.00      1.00     56962



In [25]:
# ------------------------------------------------------------------------- 
# 🔍 AUTOMATED FINTECH THRESHOLD OPTIMIZATION LOOP
# ------------------------------------------------------------------------- 
from sklearn.metrics import precision_score, recall_score, f1_score

print("\n🔎 Scanning for optimal operational risk threshold...")
print(f"{'Threshold':<12}{'Precision':<12}{'Recall':<12}{'F1-Score':<12}")
print("-" * 50)

best_f1 = 0
optimal_threshold = 0.50

# Test thresholds from 0.05 to 0.50 in steps of 0.05
for thresh in np.arange(0.05, 0.55, 0.05):
    temp_preds = (calib_probs >= thresh).astype(int)
    prec = precision_score(y_test, temp_preds, zero_division=0)
    rec = recall_score(y_test, temp_preds, zero_division=0)
    f1 = f1_score(y_test, temp_preds, zero_division=0)
    
    print(f"{thresh:<12.2f}{prec:<12.4f}{rec:<12.4f}{f1:<12.4f}")
    
    if f1 > best_f1:
        best_f1 = f1
        optimal_threshold = thresh
        
print("-" * 50)
print(f"🎯 Mathematically Optimal Threshold Found: {optimal_threshold:.2f} (F1: {best_f1:.4f})")



🔎 Scanning for optimal operational risk threshold...
Threshold   Precision   Recall      F1-Score    
--------------------------------------------------
0.05        0.4746      0.8571      0.6109      
0.10        0.7500      0.8265      0.7864      
0.15        0.8061      0.8061      0.8061      
0.20        0.8061      0.8061      0.8061      
0.25        0.8966      0.7959      0.8432      
0.30        0.8966      0.7959      0.8432      
0.35        0.9048      0.7755      0.8352      
0.40        0.9048      0.7755      0.8352      
0.45        0.9211      0.7143      0.8046      
0.50        0.9211      0.7143      0.8046      
--------------------------------------------------
🎯 Mathematically Optimal Threshold Found: 0.25 (F1: 0.8432)


In [18]:
print("\n📋 Production Classification Profile (Calibrated Metrics):")
predictions = calibrated_model.predict(X_test)
print(classification_report(y_test, predictions, target_names=['Legit', 'Fraud']))
print("==============================================================")


📋 Production Classification Profile (Calibrated Metrics):
              precision    recall  f1-score   support

       Legit       1.00      1.00      1.00     56864
       Fraud       0.93      0.58      0.72        98

    accuracy                           1.00     56962
   macro avg       0.97      0.79      0.86     56962
weighted avg       1.00      1.00      1.00     56962



In [ ]:
# Save the calibrated wrapper asset to local storage
import joblib
joblib.dump(calibrated_model, "calibrated_fraud_model.pkl")
print("💾 Calibrated structural model wrapper compiled and saved locally!")